# ILP Hazard Simulator - Bagian WAW (Write After Write)

INF-1316 Organisasi dan Arsitektur Komputer.

WAW terjadi ketika dua instruksi menulis ke register yang sama. Karena WAW adalah *name dependence*, ia dapat dihilangkan dengan *register renaming*.

Jalankan sel dari atas ke bawah.

## 1. Modul WAW (isi `hazard_waw.py`)

In [ ]:
"""
hazard_waw.py
Deteksi dan penanganan WAW (Write After Write) pada simulator ILP.

WAW (output dependence): dua instruksi menulis ke register yang SAMA.
  I(i): Rx = ...
  I(j): Rx = ...   (j > i)  -> I(j) tidak boleh selesai menulis lebih dulu dari I(i).
WAW termasuk name dependence, sehingga dapat dihilangkan dengan register renaming.
"""

from dataclasses import dataclass


@dataclass
class Instruction:
    name: str
    destination: str
    sources: list


# ---------------------------------------------------------------- deteksi
def has_raw(i1, i2):
    """RAW: i2 membaca register yang ditulis i1."""
    return i1.destination in i2.sources


def has_waw(i1, i2):
    """WAW: i1 dan i2 menulis ke register yang sama."""
    return i1.destination == i2.destination


def dependency_type(i1, i2, use_waw=True):
    """Jenis dependensi i2 terhadap i1 (i1 lebih dulu)."""
    result = []
    if has_raw(i1, i2):
        result.append("RAW")
    if use_waw and has_waw(i1, i2):
        result.append("WAW")
    return result


def get_dependencies(instrs, use_waw=True):
    """List (dari, ke, jenis, register). Dipakai untuk dependency graph."""
    edges = []
    for a in range(len(instrs)):
        for b in range(a + 1, len(instrs)):
            for t in dependency_type(instrs[a], instrs[b], use_waw):
                if t == "RAW":
                    # RAW sejati hanya ke penulis TERAKHIR sebelum pembaca
                    overwritten = any(
                        instrs[m].destination == instrs[a].destination
                        for m in range(a + 1, b)
                    )
                    if overwritten:
                        continue
                edges.append((instrs[a].name, instrs[b].name, t, instrs[a].destination))
    return edges


def show_dependencies(instrs, title, use_waw=True):
    print("\n" + "=" * 55)
    print(title)
    print("=" * 55)
    edges = get_dependencies(instrs, use_waw)
    if not edges:
        print("Tidak ada dependensi.")
    for src, dst, t, reg in edges:
        print(f"{dst} bergantung pada {src}  [{t}]  (register {reg})")
    return edges


def detect_waw_violation(instrs):
    """Simulasi jika simulator HANYA mengecek RAW: cari dua penulis register
    yang sama di cycle yang sama (urutan tulis tidak terjamin = hazard)."""
    completed, remaining, cycle, violations = set(), instrs.copy(), 0, []
    while remaining:
        cycle += 1
        this_cycle = []
        for ins in remaining:
            ok = True
            for prev in instrs:
                if prev.name == ins.name:
                    break
                if has_raw(prev, ins) and prev.name not in completed:
                    ok = False
                    break
            if ok:
                this_cycle.append(ins)
        for a in range(len(this_cycle)):
            for b in range(a + 1, len(this_cycle)):
                if has_waw(this_cycle[a], this_cycle[b]):
                    violations.append((cycle, this_cycle[a].name,
                                       this_cycle[b].name, this_cycle[a].destination))
        for ins in this_cycle:
            completed.add(ins.name)
            remaining.remove(ins)
    return violations


def final_writer(instrs):
    """Instruksi yang seharusnya menentukan nilai akhir tiap register."""
    last = {}
    for ins in instrs:
        last[ins.destination] = ins.name
    return last


# ------------------------------------------------------- register renaming
def register_renaming(instrs):
    """Beri nama baru tiap penulisan (R1 -> R1_1, R1_2, ...). WAW hilang."""
    counter, mapping, renamed = {}, {}, []
    for ins in instrs:
        new_src = [mapping.get(s, s) for s in ins.sources]
        counter[ins.destination] = counter.get(ins.destination, 0) + 1
        new_dst = f"{ins.destination}_{counter[ins.destination]}"
        mapping[ins.destination] = new_dst
        renamed.append(Instruction(ins.name, new_dst, new_src))
    return renamed


# --------------------------------------------------------------- eksekusi
def sequential_execution(instrs):
    print("\n" + "=" * 55)
    print("SEQUENTIAL EXECUTION")
    print("=" * 55)
    cycle = 0
    for ins in instrs:
        cycle += 1
        print(f"Cycle {cycle}: {ins.name} ({ins.destination} = {' + '.join(ins.sources)})")
    print(f"\nTotal Sequential Cycles: {cycle}")
    return cycle


def parallel_execution(instrs, title, use_waw=True):
    """ILP: instruksi jalan paralel bila dependensinya (RAW + WAW) sudah selesai."""
    print("\n" + "=" * 55)
    print(title)
    print("=" * 55)
    completed, remaining, cycle = set(), instrs.copy(), 0
    while remaining:
        cycle += 1
        this_cycle = []
        for ins in remaining:
            ok = True
            for prev in instrs:
                if prev.name == ins.name:
                    break
                if dependency_type(prev, ins, use_waw) and prev.name not in completed:
                    ok = False
                    break
            if ok:
                this_cycle.append(ins)
        print(f"\nCycle {cycle}:")
        for ins in this_cycle:
            print(f"  -> {ins.name}: {ins.destination} = {' + '.join(ins.sources)}")
        for ins in this_cycle:
            completed.add(ins.name)
            remaining.remove(ins)
    print(f"\nTotal Cycles: {cycle}")
    return cycle


## 2. Program contoh (isi `programs.py`)

In [ ]:
"""
programs.py
Program contoh (kumpulan instruksi) untuk menguji WAW.
Sengaja tidak memuat WAR agar yang dianalisis murni WAW.
"""


# WAW pada R1 (I1 -> I3) dan R4 (I2 -> I5). I6 membaca R1 dan R4 (RAW).
WAW_PROGRAM = [
    Instruction("I1", "R1", ["R2", "R3"]),
    Instruction("I2", "R4", ["R5", "R6"]),
    Instruction("I3", "R1", ["R7", "R8"]),     # WAW dengan I1 (R1)
    Instruction("I4", "R9", ["R10", "R11"]),
    Instruction("I5", "R4", ["R12", "R13"]),   # WAW dengan I2 (R4)
    Instruction("I6", "R14", ["R1", "R4"]),    # RAW ke I3 dan I5
]


## 3. Simulasi

In [ ]:
# Menjalankan simulasi WAW lengkap
instructions = WAW_PROGRAM

print("Instruction Set:")
for ins in instructions:
    print(f"{ins.name}: {ins.destination} = {' + '.join(ins.sources)}")

show_dependencies(instructions, "DEPENDENCY ANALYSIS (RAW + WAW)")

print("\nDETEKSI HAZARD WAW (jika hanya cek RAW)")
for cyc, a, b, reg in detect_waw_violation(instructions):
    print(f"[HAZARD] Cycle {cyc}: {a} dan {b} sama-sama menulis {reg}")

seq = sequential_execution(instructions)
waw = parallel_execution(instructions, "ILP - DENGAN PENANGANAN WAW (stall)", True)

renamed = register_renaming(instructions)
print("\nSetelah Register Renaming:")
for ins in renamed:
    print(f"{ins.name}: {ins.destination} = {' + '.join(ins.sources)}")
show_dependencies(renamed, "DEPENDENCY SETELAH RENAMING (WAW hilang)")
ren = parallel_execution(renamed, "ILP - SETELAH REGISTER RENAMING", True)

print("\nPERFORMANCE COMPARISON")
print(f"Sequential              : {seq} cycle")
print(f"ILP + WAW stall         : {waw} cycle (speedup {seq/waw:.2f}x)")
print(f"ILP + Register Renaming : {ren} cycle (speedup {seq/ren:.2f}x)")


## 4. Dependency Graph

In [ ]:
# Dependency graph (WAW = merah putus-putus, RAW = hitam)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle

POS = {"I1": (1, 3), "I2": (1, 1), "I4": (1, -1), "I3": (4, 3), "I5": (4, 1), "I6": (7, 2)}

def draw(ax, instrs, title):
    for s, d, t, reg in get_dependencies(instrs):
        col = "#d62728" if t == "WAW" else "#1f1f1f"
        (x1, y1), (x2, y2) = POS[s], POS[d]
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=20,
                     color=col, linestyle="--" if t == "WAW" else "-", linewidth=2.2,
                     shrinkA=34, shrinkB=34))
        ax.text((x1+x2)/2, (y1+y2)/2+0.18, f"{t} ({reg})", color=col, ha="center",
                fontsize=9, fontweight="bold", bbox=dict(fc="white", ec="none", pad=1))
    for ins in instrs:
        x, y = POS[ins.name]
        ax.add_patch(Circle((x, y), 0.55, fc="#dbe9f8", ec="#1f4e79", lw=1.8))
        ax.text(x, y, ins.name, ha="center", va="center", fontsize=12, fontweight="bold")
        ax.text(x, y-0.85, f"{ins.destination} = {' + '.join(ins.sources)}", ha="center", fontsize=8.5)
    ax.set_xlim(-0.2, 8.2); ax.set_ylim(-2.1, 4.2); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=12, fontweight="bold")

fig, axs = plt.subplots(2, 1, figsize=(9, 9.6))
draw(axs[0], WAW_PROGRAM, "Dependency Graph WAW (sebelum renaming)")
draw(axs[1], register_renaming(WAW_PROGRAM), "Setelah Register Renaming (WAW hilang)")
plt.tight_layout(); plt.show()
